
# 01 — Environment & Intelligence
**Agentic X | Studio 67**

### Overview
This notebook sets up the simulated event environment, validates **150 test cases and 450 solution options**, and tests the Creative Agent using Gemini.

### Creative Agent
Evaluates each proposed solution based on **incident relevance, event quality, and audience experience**. It compares the solution's quality score with the minimum quality requirement and returns an **Accepted/Rejected** decision with a reason.

### Related Python Files
- `core/environment.py` — Dataset loading, validation, and ground-truth protection.
- `agents/creative.py` — Creative Agent logic and Gemini evaluation.
- `core/models.py` — Shared agent evaluation and communication models.

### Gemini API Setup
1. Get an API key: https://aistudio.google.com/apikey
2. Enter the key when prompted in the notebook.
3. Run the remaining cells in order.

**Note:** The API key must not be saved in the notebook or uploaded to GitHub.

Cell 2 — Connect to GitHub

In [4]:

from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/ProjectsSpaceHub/AgenticX.git"
ROOT = Path("/content/AgenticX")

if not ROOT.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(ROOT)],
        check=True
    )

os.chdir(ROOT)

print("Project directory:", Path.cwd())


Project directory: /content/AgenticX


Cell 3 — Install Requirements

In [5]:

import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements.txt"
    ],
    check=True
)

print("Dependencies installed successfully")


Dependencies installed successfully


Cell 4 — Import Libraries

In [6]:

import pandas as pd
from IPython.display import display

from core.environment import (
    DEFAULT_DATA,
    ENVIRONMENT_DESCRIPTION,
    HIDDEN_OPTION_FIELDS,
    HIDDEN_CASE_FIELDS,
    load_dataset,
    get_case
)

from agents.creative import CreativeAgent

print("All imports completed")


All imports completed


Cell 5 — Load the Dataset


In [27]:

cases_df, options_df = load_dataset()

print("Total scenarios:", len(cases_df))
print("Total solution options:", len(options_df))
print("Unique cases:", cases_df["case_id"].nunique())

display(cases_df[[
    "case_id", "scenario", "incident",
    "initial_budget_sar", "time_left_min",
    "required_quality_min"
]].head())

display(options_df[[
    "option_id", "case_id", "option_name",
    "cost_sar", "duration_min", "quality_score"
]].head())


Total scenarios: 150
Total solution options: 450
Unique cases: 150


,case_id,scenario,incident,initial_budget_sar,time_left_min,required_quality_min
0,SCR-001,Screen Failure,Main LED screen failed during final rehearsal,8500,100,60
1,SCR-002,Screen Failure,LED wall lost signal before keynote,2400,60,50
2,SCR-003,Screen Failure,Display controller failure before exhibition o...,7220,90,50
3,SCR-004,Screen Failure,Screen power module stopped working,14000,150,50
4,SCR-005,Screen Failure,Main stage screen shows intermittent blackout,11000,150,55


,option_id,case_id,option_name,cost_sar,duration_min,quality_score
0,SCR-001-O1,SCR-001,Use backup LED,1290,40,89
1,SCR-001-O2,SCR-001,Rent replacement screen,5920,70,89
2,SCR-001-O3,SCR-001,Continue with projection,1210,25,60
3,SCR-002-O1,SCR-002,Use backup LED,1230,40,75
4,SCR-002-O2,SCR-002,Rent replacement screen,6300,80,96


Cell 6 — Validate the Dataset

In [8]:

assert len(cases_df) == 150
assert len(options_df) == 450

assert cases_df["case_id"].is_unique
assert options_df["option_id"].is_unique

assert not cases_df["case_id"].isna().any()
assert not options_df["case_id"].isna().any()

assert set(options_df["case_id"]).issubset(
    set(cases_df["case_id"])
)

options_per_case = options_df.groupby("case_id").size()

assert (options_per_case == 3).all()

print("Dataset validation passed")
print("\nScenario distribution:")
print(cases_df["scenario"].value_counts())


Dataset validation passed

Scenario distribution:
scenario
Screen Failure     50
Budget Overrun     50
Safety Conflict    50
Name: count, dtype: int64


Cell 7 — Define the Simulated Environment.

In [9]:

environment = {
    "name": "Studio 67",
    "domain": "Event Crisis Management",
    "agents": [
        "Creative",
        "Finance",
        "Operations",
        "Safety"
    ],
    "resources": [
        "Budget",
        "Time",
        "Crew",
        "Equipment",
        "Venue"
    ],
    "scenario_types": sorted(
        cases_df["scenario"].unique().tolist()
    )
}

for key, value in environment.items():
    print(f"{key}: {value}")


name: Studio 67
domain: Event Crisis Management
agents: ['Creative', 'Finance', 'Operations', 'Safety']
resources: ['Budget', 'Time', 'Crew', 'Equipment', 'Venue']
scenario_types: ['Budget Overrun', 'Safety Conflict', 'Screen Failure']


Cell 8 — Environment Classification

In [10]:
classification = {
    "Observability": (
        "Partially observable / inaccessible",
        "Agents receive selected incident and resource details, not all relevant real-world facts."
    ),
    "Determinism": (
        "Stochastic overall; deterministic rule checks",
        "Gemini judgments may vary; Finance and Safety checks follow fixed rules."
    ),
    "Dynamics": (
        "Dynamic between rounds",
        "Time remaining decreases during negotiation; recorded solution properties stay fixed."
    ),
    "State Space": (
        "Discrete decisions with continuous attributes",
        "Choices and ACL messages are discrete; time, cost and quality are numerical."
    )
}
classification_df = pd.DataFrame(
    [{"Property": k, "Classification": v[0], "Justification": v[1]}
     for k,v in classification.items()]
)
display(classification_df)

,Property,Classification,Justification
0,Observability,Partially observable / inaccessible,Agents receive selected incident and resource ...
1,Determinism,Stochastic overall; deterministic rule checks,Gemini judgments may vary; Finance and Safety ...
2,Dynamics,Dynamic between rounds,Time remaining decreases during negotiation; r...
3,State Space,Discrete decisions with continuous attributes,"Choices and ACL messages are discrete; time, c..."


Cell 9 — Creative Agent PEAS

In [11]:
creative_peas = {
    "Performance Measure": (
        "Incident relevance, solution quality, "
        "and audience experience"
    ),
    "Environment": (
        "Studio 67 event crisis simulation"
    ),
    "Actuators": (
        "Evaluate solutions and return structured "
        "approval, score, and reasoning"
    ),
    "Sensors": (
        "Incident description, available solutions, "
        "and quality requirements"
    )
}

peas_df = pd.DataFrame(
    creative_peas.items(),
    columns=["PEAS Component", "Description"]
)

display(peas_df)

print("Agent Type: Deliberative")
print(
    "Justification: Creative uses LLM reasoning to "
    "evaluate alternatives before providing its assessment."
)

,PEAS Component,Description
0,Performance Measure,"Incident relevance, solution quality, and audi..."
1,Environment,Studio 67 event crisis simulation
2,Actuators,Evaluate solutions and return structured appro...
3,Sensors,"Incident description, available solutions, and..."


Agent Type: Deliberative
Justification: Creative uses LLM reasoning to evaluate alternatives before providing its assessment.


Cell 10 — Select a Test Scenario

In [12]:

CASE_ID = "SCR-001"

case, options = get_case(CASE_ID)

print("Case ID:", case["case_id"])
print("Scenario:", case["scenario"])
print("Incident:", case["incident"])
print("Available budget:", case["initial_budget_sar"])
print("Time remaining:", case["time_left_min"])

print("\nAvailable solutions:")

display(
    pd.DataFrame(options)[
        [
            "option_id",
            "option_name",
            "cost_sar",
            "duration_min",
            "quality_score"
        ]
    ]
)


Case ID: SCR-001
Scenario: Screen Failure
Incident: Main LED screen failed during final rehearsal
Available budget: 8500
Time remaining: 100

Available solutions:


,option_id,option_name,cost_sar,duration_min,quality_score
0,SCR-001-O1,Use backup LED,1290,40,89
1,SCR-001-O2,Rent replacement screen,5920,70,89
2,SCR-001-O3,Continue with projection,1210,25,60


Cell 11 — Prevent Evaluation Data Leakage

In [13]:

assert "case_profile" not in case

for option in options:
    assert "addresses_incident" not in option

    for field in HIDDEN_OPTION_FIELDS:
        assert field not in option

print("No evaluation ground-truth labels exposed")
print("Creative will assess the options independently")


No evaluation ground-truth labels exposed
Creative will assess the options independently


### Verify the conditions and option-specific decision inputs
These are **original Excel values**. A live UI may override adjustable conditions; agents must receive current values for a what-if test. The original options remain unchanged.

In [14]:
import math
required_case_fields = ["case_id", "scenario", "initial_budget_sar", "time_left_min", "required_quality_min"]
required_option_fields = ["option_id", "cost_sar", "duration_min", "quality_score", "safety_ok", "permit_ok", "mitigation_verified", "risk_score_after_mitigation"]
assert all(field in case for field in required_case_fields)
assert all(all(field in option for field in required_option_fields) for option in options)
assert math.isfinite(float(case["initial_budget_sar"]))
assert math.isfinite(float(case["time_left_min"]))
assert 0 <= float(case["required_quality_min"]) <= 100

display(pd.DataFrame([{
    "case_id": case["case_id"],
    "original_budget_sar": case["initial_budget_sar"],
    "original_time_min": case["time_left_min"],
    "minimum_quality": case["required_quality_min"]
}]))
display(pd.DataFrame(options)[required_option_fields])
print("Option decision inputs verified; no ground-truth outcome columns are passed to agents.")

,case_id,original_budget_sar,original_time_min,minimum_quality
0,SCR-001,8500,100,60


,option_id,cost_sar,duration_min,quality_score,safety_ok,permit_ok,mitigation_verified,risk_score_after_mitigation
0,SCR-001-O1,1290,40,89,True,True,True,1
1,SCR-001-O2,5920,70,89,True,True,True,1
2,SCR-001-O3,1210,25,60,True,True,True,3


Option decision inputs verified; no ground-truth outcome columns are passed to agents.


Cell 12 — Configure LLM Access

In [15]:
# Dependencies were installed from the repository requirements.txt above.
# Avoid upgrading a library mid-notebook and creating version inconsistencies.
from importlib.metadata import version
print("langchain-google-genai:", version("langchain-google-genai"))

langchain-google-genai: 4.5.0


In [16]:
import os
from getpass import getpass

if not os.getenv("GOOGLE_API_KEY"):
    api_key = getpass("Enter your Gemini API key: ").strip()
    if not api_key:
        raise RuntimeError("Gemini API key required")
    os.environ["GOOGLE_API_KEY"] = api_key

os.environ.setdefault("AGENTICX_MODEL", "gemini-3.5-flash-lite")
print("Gemini configured. Model:", os.environ["AGENTICX_MODEL"])

Enter your Gemini API key: ··········
Gemini configured. Model: gemini-3.5-flash-lite


In [17]:
# IMPORTANT: the shared source file agents/creative.py must not be overwritten.
# Earlier notebook versions wrote a legacy agent implementation here, disabling
# the filtering of agent-specific inputs and causing inconsistent evaluations.
from agents.creative import CreativeAgent
import inspect

assert "required_quality_min" in CreativeAgent.CASE_FIELDS
assert "quality_score" in CreativeAgent.OPTION_FIELDS
assert "cost_sar" not in CreativeAgent.OPTION_FIELDS
assert "safety_ok" not in CreativeAgent.OPTION_FIELDS
print("Creative implementation:", inspect.getfile(CreativeAgent))

Creative implementation: /content/AgenticX/agents/creative.py


Cell 13 — Initialize Creative Agent

In [18]:
import importlib
import agents.creative as creative_module

importlib.reload(creative_module)
creative_agent = creative_module.CreativeAgent()
print("Creative Agent initialized from repository source")

Creative Agent initialized from repository source


Cell 14 — Evaluate the Three Solutions

In [19]:

creative_results = []

for option in options:

    print("Evaluating:", option["option_name"])

    evaluation = creative_agent.evaluate(
        case,
        option
    )

    creative_results.append(
        evaluation.model_dump()
    )

results_df = pd.DataFrame(creative_results)

display(results_df)


Evaluating: Use backup LED
Evaluating: Rent replacement screen
Evaluating: Continue with projection


,agent,option_id,approved,score,reason,blocking
0,Creative,SCR-001-O1,True,89.0,Solution quality: 89/100. Minimum required: 60...,False
1,Creative,SCR-001-O2,True,89.0,Solution quality: 89/100. Minimum required: 60...,False
2,Creative,SCR-001-O3,True,60.0,Solution quality: 60/100. Minimum required: 60...,False


Cell 15 — Validate Creative Outputs

In [20]:

assert len(results_df) == len(options)

assert results_df["option_id"].is_unique

assert results_df["score"].between(0, 100).all()

assert results_df["reason"].notna().all()

assert (
    results_df["agent"] == "Creative"
).all()

print("Creative output validation passed")


Creative output validation passed


Cell 16 — Display Creative Recommendations


In [21]:

ranked_results = results_df.sort_values(
    by="score",
    ascending=False
)

display(
    ranked_results[
        [
            "option_id",
            "approved",
            "score",
            "reason"
        ]
    ]
)

approved_results = ranked_results[
    ranked_results["approved"]
]

if approved_results.empty:
    print("Creative did not approve any solution.")
else:
    best = approved_results.iloc[0]

    print("Creative preferred option:", best["option_id"])
    print("Creative score:", best["score"])

print(
    "\nNote: Final selection requires Finance, "
    "Operations and Safety approval."
)


,option_id,approved,score,reason
0,SCR-001-O1,True,89.0,Solution quality: 89/100. Minimum required: 60...
1,SCR-001-O2,True,89.0,Solution quality: 89/100. Minimum required: 60...
2,SCR-001-O3,True,60.0,Solution quality: 60/100. Minimum required: 60...


Creative preferred option: SCR-001-O1
Creative score: 89.0

Note: Final selection requires Finance, Operations and Safety approval.


Cell 17 — Save Results

In [22]:

from pathlib import Path

output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

output_path = output_dir / "creative_evaluations.csv"

results_df.to_csv(
    output_path,
    index=False
)

print("Results saved to:", output_path)


Results saved to: results/creative_evaluations.csv


In [23]:
additional_results = []

scenario_types = [
    "Budget Overrun",
    "Safety Conflict"
]

for scenario_type in scenario_types:

    matching = cases_df[
        cases_df["scenario"] == scenario_type
    ]

    assert not matching.empty, f"No cases for {scenario_type}"

    test_case_id = matching.iloc[0]["case_id"]
    test_case, test_options = get_case(test_case_id)

    # Evaluate one real option from each additional scenario.
    option = test_options[0]

    evaluation = creative_agent.evaluate(
        test_case,
        option
    )

    additional_results.append({
        "scenario": scenario_type,
        **evaluation.model_dump()
    })

additional_df = pd.DataFrame(additional_results)

display(additional_df)

assert len(additional_df) == 2
assert additional_df["score"].between(0, 100).all()
assert additional_df["reason"].str.strip().ne("").all()

print("Additional scenario evaluations completed")

,scenario,agent,option_id,approved,score,reason,blocking
0,Budget Overrun,Creative,BDG-001-O1,True,64.0,Solution quality: 64/100. Minimum required: 55...,False
1,Safety Conflict,Creative,SFT-001-O1,False,0.0,Solution quality: 67/100. Minimum required: 55...,False


Additional scenario evaluations completed


Cell 18 — Creative Quality Threshold Test


In [24]:

# Test Creative with a higher minimum quality

test_case = case.copy()
test_case["required_quality_min"] = 95

evaluation = creative_agent.evaluate(
    test_case,
    options[0]
)

print("Option quality:", options[0]["quality_score"])
print("Required quality:", test_case["required_quality_min"])
print("Approved:", evaluation.approved)
print("Reason:", evaluation.reason)

assert evaluation.approved is False
print("Quality threshold test passed!")


Option quality: 89
Required quality: 95
Approved: False
Reason: Solution quality: 89/100. Minimum required: 95/100. The solution is below the minimum quality requirement. Rejected.
Quality threshold test passed!


Cell 19 — Python Files Verification

In [25]:

from pathlib import Path
import subprocess
import importlib
import inspect

ROOT = Path("/content/AgenticX")

# Latest verified GitHub file versions
expected = {
    "core/environment.py": "39e7321935bc4e59dbe0e474cdeedf69b4b5e872",
    "agents/creative.py": "0876552ad7c5994e8b2327ead36883674b47e9c3",
    "core/models.py": "aeb1a0fc29984ed54d4252243ef1718808cafb80"
}

for file, github_sha in expected.items():
    local_file = ROOT / file

    if not local_file.exists():
        print(f"❌ {file}: File missing")
        continue

    local_sha = subprocess.check_output(
        ["git", "hash-object", str(local_file)],
        text=True
    ).strip()

    if local_sha == github_sha:
        print(f"✅ {file}: UP TO DATE")
    else:
        print(f"❌ {file}: DIFFERENT FROM GITHUB")

# Reload the current files into Python
import core.environment as env
import agents.creative as creative

importlib.reload(env)
importlib.reload(creative)

print("\nCreative source:", inspect.getfile(creative.CreativeAgent))
print("Environment source:", inspect.getfile(env))


✅ core/environment.py: UP TO DATE
✅ agents/creative.py: UP TO DATE
✅ core/models.py: UP TO DATE

Creative source: /content/AgenticX/agents/creative.py
Environment source: /content/AgenticX/core/environment.py


### Cell 20 — Creative Relevance Test

This test checks whether the Creative Agent considers incident relevance in addition to the minimum quality requirement. It displays the recorded evaluation of a safety-conflict solution and explains why the agent rejected it despite meeting the quality threshold.

In [26]:

# Inspect the Creative decision for SFT-001-O1

sft_case, sft_options = get_case("SFT-001")

sft_result = additional_df.loc[
    additional_df["option_id"] == "SFT-001-O1"
].iloc[0]

print("Scenario:", sft_case["scenario"])
print("Option:", sft_options[0]["option_name"])
print("Quality:", sft_options[0]["quality_score"])
print("Minimum quality:", sft_case["required_quality_min"])
print("Creative approved:", sft_result["approved"])
print("Creative reason:", sft_result["reason"])

assert (
    sft_options[0]["quality_score"]
    >= sft_case["required_quality_min"]
)

assert not bool(sft_result["approved"])

print("Creative relevance test passed!")


Scenario: Safety Conflict
Option: Leave stage equipment blocking emergency exit
Quality: 67
Minimum quality: 55
Creative approved: False
Creative reason: Solution quality: 67/100. Minimum required: 55/100. The quality requirement is met. Leaving stage equipment blocking the emergency exit does not address the safety incident and severely compromises audience experience and event quality. Creative assessment rejected.
Creative relevance test passed!


## Conclusion

This notebook prepared and validated the Studio 67 event crisis dataset, described the simulated environment, and documented the Creative Agent using PEAS.

The Creative Agent uses a real LLM to evaluate alternative solutions based on incident relevance and event quality.

Its evaluations are structured and validated before being passed to the multi-agent coordination process.

Final decisions require collaboration with Finance, Operations, and Safety.